# Modelo de treinamento inicial

Vamos iniciar o processo de treinamento do modelo inicial, para isso começamos com o download dos dados, e verificação dos mesmos, entender o modelo dos dados e como são distribuidos, para realizar o tratamento dos dados e incorporar no modelo.

## configurando acesso ao Kaggle

usando modelo Json para configuração da chave de api, e para obter acesso aos dados

In [10]:
import json
import os

# Suas credenciais obtidas no arquivo kaggle.json do Kaggle
kaggle_creds = {
    "username": "marcknero",
    "key": "b8dac997627d715601ceb6f0b8ddcde2"
}

# Cria o diretório e o arquivo kaggle.json com as permissões corretas
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_creds, f)

os.chmod('/root/.kaggle/kaggle.json', 0o600)
print("Credenciais do Kaggle configuradas com sucesso!")

Credenciais do Kaggle configuradas com sucesso!


## baixando o dataset

In [11]:
import kagglehub


print('Baixando o dataset do Olist via Kagglehub...')

#faz o download e retorna o diretorio onde os arquivos foram salvos
dataset_path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")
print(f'Arquivos salvos no diretório: {dataset_path}\n')

#lista os arquivos disponíveis no diretório
files = os.listdir(dataset_path)
print(f'Arquivos disponíveis no diretório: ')
for f in files:
  print(f'- {f}')


Baixando o dataset do Olist via Kagglehub...
Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
Arquivos salvos no diretório: /kaggle/input/brazilian-ecommerce

Arquivos disponíveis no diretório: 
- olist_customers_dataset.csv
- olist_sellers_dataset.csv
- olist_order_reviews_dataset.csv
- olist_order_items_dataset.csv
- olist_products_dataset.csv
- olist_geolocation_dataset.csv
- product_category_name_translation.csv
- olist_orders_dataset.csv
- olist_order_payments_dataset.csv


## verificando os dados e formatos



In [15]:
import pandas as pd

# carregar tabelas para inspeção
print('Carregando tabelas para inspeção...')

orders = pd.read_csv(os.path.join(dataset_path, 'olist_orders_dataset.csv'))
items = pd.read_csv(os.path.join(dataset_path, 'olist_order_items_dataset.csv'))
customers = pd.read_csv(os.path.join(dataset_path, 'olist_customers_dataset.csv'))
products = pd.read_csv(os.path.join(dataset_path, 'olist_products_dataset.csv'))

print('\n --- 1. Status dos pedidos (orders) ---')
print(orders['order_status'].value_counts(dropna=False))

print('\n --- 2. Verificando valores nulos (orders) ---')
print(orders.isnull().sum())

print('\n --- 3. Analizando Variável Alvo (atrasos) ---')
#filtrar pedidos entregues
delivered = orders[orders['order_status']=='delivered'].copy()
#converte coluda de datas
delivered['order_delivered_customer_date'] = pd.to_datetime(delivered['order_delivered_customer_date'])
delivered['order_estimated_delivery_date'] = pd.to_datetime(delivered['order_estimated_delivery_date'])
#cria coluna de atraso para ver a proporção
delivered['is_late'] = (delivered['order_delivered_customer_date'] > delivered['order_estimated_delivery_date']).astype(int)
late_counts = delivered['is_late'].value_counts(normalize=True)*100
print(f'Pedidos entregues no prazo: {late_counts.get(0,0):.2f}%')
print(f'Pedidos entregues com atraso: {late_counts.get(1,0):.2f}%')

print(f'\n Total de pedidos analizados na base limpa de entregas: {len(delivered)}')


#sugestao de uso para separação de modelo funcional, caso necessario retreinamento para aplicação, separando o processo em componentes individuais
# apenas se for decidido que o modelo ira passar por retreinamentos periodicos para melhorar o feedback com dados adicionados posteriormente
# if __name__ == '__main__':
#   dataset_path = download_and_inspect_data()


Carregando tabelas para inspeção...

 --- 1. Status dos pedidos (orders) ---
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

 --- 2. Verificando valores nulos (orders) ---
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

 --- 3. Analizando Variável Alvo (atrasos) ---
Pedidos entregues no prazo: 91.89%
Pedidos entregues com atraso: 8.11%

 Total de pedidos analizados na base limpa de entregas: 96478


## Engenharia de atributos e Cruzamento de dados

Para que o modelo obtenha o melhor rendimento em seu treinamento e consiga entrega as predições com a menor falha possivel.
Vamos tratar os dados obtidos do dataset, transformando os dados brutos e esparsos em uma unica tabela coesa, para o modelo, unindo informações de preço, frete dimensoes fisicas do produto, estados de origem e destino, distancia geografica real e padroes temporais de compra
E por fim realizaremos uma limpesa final, selecinando esclusivamente as colunas preditoras e a variavel alvo (is_late) para eliminar eventuais valroes nulos residuias

In [18]:
import numpy as np

def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

print("Carregando tabelas complementares...")
geolocation = pd.read_csv(os.path.join(dataset_path, 'olist_geolocation_dataset.csv'))
sellers = pd.read_csv(os.path.join(dataset_path, 'olist_sellers_dataset.csv'))

# 1. Agregando itens, para avaliar o tamanho real do pedido com todos os produtos da mesma entrega
item_agg = items.groupby('order_id').agg({
    'price': 'sum',
    'freight_value': 'sum',
    'product_id': 'first',
    'seller_id': 'first'
}).reset_index()

# 2. Juntando dados de produtos, agregando de forma a manter o controle por lote
product_info = products[['product_id', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']]
item_agg = item_agg.merge(product_info, on='product_id', how='left')

# 3. Consolidando com pedidos entregues
df_model = delivered.merge(item_agg, on='order_id', how='inner')

# 4. Cruzando clientes e vendedores para capturar estados e CEPs
df_model = df_model.merge(customers[['customer_id', 'customer_zip_code_prefix', 'customer_state']], on='customer_id', how='left')
df_model = df_model.merge(sellers[['seller_id', 'seller_zip_code_prefix', 'seller_state']], on='seller_id', how='left')

# 5. Calculando distâncias geográficas via Haversine
geo_mean = geolocation.groupby('geolocation_zip_code_prefix').agg({
    'geolocation_lat': 'mean',
    'geolocation_lng': 'mean'
}).reset_index()

df_model = df_model.merge(geo_mean, left_on='customer_zip_code_prefix', right_on='geolocation_zip_code_prefix', how='left')
df_model.rename(columns={'geolocation_lat': 'cust_lat', 'geolocation_lng': 'cust_lng'}, inplace=True)
df_model.drop(columns=['geolocation_zip_code_prefix'], inplace=True, errors='ignore')

df_model = df_model.merge(geo_mean, left_on='seller_zip_code_prefix', right_on='geolocation_zip_code_prefix', how='left')
df_model.rename(columns={'geolocation_lat': 'sell_lat', 'geolocation_lng': 'sell_lng'}, inplace=True)
df_model.drop(columns=['geolocation_zip_code_prefix'], inplace=True, errors='ignore')

df_model['distance_km'] = haversine(df_model['sell_lat'], df_model['sell_lng'], df_model['cust_lat'], df_model['cust_lng'])

# 6. Extraindo atributos temporais
df_model['order_purchase_timestamp'] = pd.to_datetime(df_model['order_purchase_timestamp'])
df_model['purchase_dayofweek'] = df_model['order_purchase_timestamp'].dt.dayofweek
df_model['purchase_month'] = df_model['order_purchase_timestamp'].dt.month

# 7. Seleção final das colunas do modelo
features_to_keep = [
    'price', 'freight_value', 'product_weight_g',
    'product_length_cm', 'product_height_cm', 'product_width_cm',
    'customer_state', 'seller_state', 'distance_km',
    'purchase_dayofweek', 'purchase_month', 'is_late'
]

df_model = df_model[features_to_keep].dropna().copy()
print(f"Base tratada e pronta para ML! Total de registros válidos: {len(df_model)}")

print('--- Amostra de dados tratados (5 primeiras linhas) ---')
display(df_model.head())

Carregando tabelas complementares...
Base tratada e pronta para ML! Total de registros válidos: 95984
--- Amostra de dados tratados (5 primeiras linhas) ---


,price,freight_value,product_weight_g,product_length_cm,product_height_cm,product_width_cm,customer_state,seller_state,distance_km,purchase_dayofweek,purchase_month,is_late
0,29.99,8.72,500.0,19.0,8.0,13.0,SP,SP,18.576110,0,10,0
1,118.70,22.76,400.0,19.0,13.0,19.0,BA,SP,851.495069,1,7,0
2,159.90,19.22,420.0,24.0,19.0,21.0,GO,SP,514.410666,2,8,0
3,45.00,27.20,450.0,30.0,10.0,20.0,RN,MG,1822.226336,5,11,0
4,19.90,8.72,250.0,51.0,15.0,15.0,SP,SP,29.676625,1,2,0


In [19]:
import joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

print("1. Separando os dados em Atributos (X) e Variável Alvo (y)...")
# X contém todas as características preditoras (preço, peso, distância, estados, etc.)
X = df_model.drop(columns=['is_late'])
# y contém a resposta que queremos prever (1 para atrasado, 0 para no prazo)
y = df_model['is_late']

print("2. Dividindo os dados entre conjuntos de Treino (80%) e Teste (20%)...")
# Usamos 'stratify=y' para garantir que a proporção de pedidos atrasados seja igual em ambos os conjuntos
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("3. Definindo os tipos de variáveis para o pré-processamento...")
# Variáveis numéricas que precisam ser normalizadas (escala padronizada)
numeric_features = [
    'price', 'freight_value', 'product_weight_g',
    'product_length_cm', 'product_height_cm', 'product_width_cm',
    'distance_km', 'purchase_dayofweek', 'purchase_month'
]
# Variáveis categóricas (estados) que precisam ser transformadas em colunas binárias (One-Hot Encoding)
categorical_features = ['customer_state', 'seller_state']

print("4. Configurando o pré-processador das colunas...")
# O ColumnTransformer aplica o StandardScaler nas numéricas e o OneHotEncoder nas categóricas
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

print("5. Construindo o Pipeline de Machine Learning...")
# O Pipeline une o pré-processamento e o modelo em um único fluxo automatizado, evitando erros na aplicação web
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=100,             # Quantidade de árvores de decisão na floresta
        max_depth=14,                 # Limite de profundidade para evitar que o modelo decore os dados (overfitting)
        random_state=42,              # Semente para garantir resultados reprodutíveis
        class_weight='balanced',      # Essencial! Penaliza erros na classe minoritária (atrasos) compensando o desbalanceamento
        n_jobs=-1                     # Utiliza todos os núcleos disponíveis do processador para acelerar o treino
    ))
])

print("6. Iniciando o treinamento do modelo com os dados de treino...")
pipeline.fit(X_train, y_train)

print("7. Avaliando o desempenho do modelo no conjunto de teste...")
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

# Exibe o relatório detalhado (Precision, Recall e F1-Score)
print("\nRelatório de Classificação:")
print(classification_report(y_test, y_pred))

# Exibe a métrica ROC-AUC (mede a capacidade do modelo de separar previsões certas de erradas)
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_proba):.4f}")

# print("\n8. Salvando o modelo treinado para uso posterior...")
# model_filename = "delivery_delay_model.joblib"
# joblib.dump(pipeline, model_filename)
# print(f"Modelo salvo com sucesso no arquivo: '{model_filename}'!")

1. Separando os dados em Atributos (X) e Variável Alvo (y)...
2. Dividindo os dados entre conjuntos de Treino (80%) e Teste (20%)...
3. Definindo os tipos de variáveis para o pré-processamento...
4. Configurando o pré-processador das colunas...
5. Construindo o Pipeline de Machine Learning...
6. Iniciando o treinamento do modelo com os dados de treino...
7. Avaliando o desempenho do modelo no conjunto de teste...

Relatório de Classificação:
              precision    recall  f1-score   support

           0       0.95      0.87      0.91     17642
           1       0.23      0.43      0.30      1555

    accuracy                           0.83     19197
   macro avg       0.59      0.65      0.60     19197
weighted avg       0.89      0.83      0.86     19197

ROC-AUC Score: 0.7254
